<a href="https://colab.research.google.com/github/the-lizardking/ict-trading-bot/blob/main/notebooks/setup/render_env_from_drive_master.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Render Lean .env File from Google Drive Master Secrets

This notebook decrypts your encrypted master secrets file from Google Drive
and generates a small, lean `.env` file for one runtime profile.

**Your secrets are never printed to screen.**

---

## What this notebook does

| Step | What happens |
|---|---|
| 1 | Connects to your Google Drive |
| 2 | Checks that `master-secrets.sops.yaml` and `age-keys.txt` exist |
| 3 | Installs SOPS and age (if not already installed) |
| 4 | Clones or updates the repo |
| 5 | Runs `render_env_from_master.py` to generate the env file |
| 6 | Shows the output path and variable names (no values) |

---

## Before you start

- [ ] You have run `encrypt_google_drive_master_secrets.ipynb` at least once
- [ ] `My Drive / ICT_Bot_Secrets / master-secrets.sops.yaml` exists
- [ ] `My Drive / ICT_Bot_Secrets / age-keys.txt` exists
- [ ] You have deleted the plaintext `master-secrets.yaml` from Drive

---
## Step 1 — Connect to Google Drive

Run the cell below. A pop-up will appear asking you to sign in to your Google
account and allow Colab to access Drive.

In [1]:
from google.colab import drive

drive.mount('/content/drive')
print('Google Drive connected.')

Mounted at /content/drive
Google Drive connected.


---
## Step 2 — Check that required files exist

This cell verifies that `master-secrets.sops.yaml` and `age-keys.txt` are
present in `My Drive / ICT_Bot_Secrets`.

> If you used a different folder name, change `SECRETS_DIR` in the cell below.

In [2]:
import os, sys

# ── Change only if you used a different folder name in Drive ──────
SECRETS_DIR = '/content/drive/MyDrive/ICT_Bot_Secrets'
# ─────────────────────────────────────────────────────────────────

MASTER_FILE  = os.path.join(SECRETS_DIR, 'master-secrets.sops.yaml')
AGE_KEY_FILE = os.path.join(SECRETS_DIR, 'age-keys.txt')

errors = []

if not os.path.isdir(SECRETS_DIR):
    errors.append(f'Folder not found: {SECRETS_DIR}')
else:
    print(f'[OK]  Folder found: {SECRETS_DIR}')

if not os.path.isfile(MASTER_FILE):
    errors.append(
        f'Encrypted file not found: {MASTER_FILE}\n'
        '       Run encrypt_google_drive_master_secrets.ipynb first.'
    )
else:
    print(f'[OK]  master-secrets.sops.yaml found ({os.path.getsize(MASTER_FILE)} bytes)')

if not os.path.isfile(AGE_KEY_FILE):
    errors.append(
        f'Age key file not found: {AGE_KEY_FILE}\n'
        '       Run encrypt_google_drive_master_secrets.ipynb first to create this key.'
    )
else:
    print(f'[OK]  age-keys.txt found')

if errors:
    print()
    for e in errors:
        print(f'ERROR: {e}')
    sys.exit(1)

print()
print('All required files are present. Proceed to Step 3.')

[OK]  Folder found: /content/drive/MyDrive/ICT_Bot_Secrets
[OK]  master-secrets.sops.yaml found (14477 bytes)
[OK]  age-keys.txt found

All required files are present. Proceed to Step 3.


---
## Step 3 — Install SOPS and age

These are the free, open-source tools used to decrypt the master secrets file.
If they are already installed from a previous run, this step is skipped automatically.

In [3]:
import subprocess, sys, shutil

AGE_VERSION  = 'v1.1.1'
SOPS_VERSION = 'v3.8.1'

def shell(cmd, check=True):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.returncode != 0:
        if result.stderr.strip():
            print('Error:', result.stderr.strip())
        if check:
            print('Installation failed. Please run this cell again.')
            sys.exit(result.returncode)
    return result

# age
if shutil.which('age'):
    print('age already installed:', shell('age --version', check=False).stdout.strip())
else:
    print(f'Installing age {AGE_VERSION} ...')
    shell(f'wget -q https://github.com/FiloSottile/age/releases/download/{AGE_VERSION}/age-{AGE_VERSION}-linux-amd64.tar.gz -O /tmp/age.tar.gz')
    shell('tar -xzf /tmp/age.tar.gz -C /tmp/')
    shell('cp /tmp/age/age /tmp/age/age-keygen /usr/local/bin/')
    shell('chmod +x /usr/local/bin/age /usr/local/bin/age-keygen')
    print('age installed:', shell('age --version', check=False).stdout.strip())

# sops
if shutil.which('sops'):
    print('sops already installed:', shell('sops --version', check=False).stdout.strip())
else:
    print(f'Installing SOPS {SOPS_VERSION} ...')
    shell(f'wget -q https://github.com/getsops/sops/releases/download/{SOPS_VERSION}/sops-{SOPS_VERSION}.linux.amd64 -O /usr/local/bin/sops')
    shell('chmod +x /usr/local/bin/sops')
    print('sops installed:', shell('sops --version', check=False).stdout.strip())

print()
print('Tools ready. Proceed to Step 4.')

Installing age v1.1.1 ...
v1.1.1
age installed: v1.1.1
Installing SOPS v3.8.1 ...
sops 3.8.1
[info] a new version of sops (v3.12.2) is available, you can update by visiting: https://github.com/getsops/sops/releases/tag/v3.12.2
sops installed: sops 3.8.1
[info] a new version of sops (v3.12.2) is available, you can update by visiting: https://github.com/getsops/sops/releases/tag/v3.12.2

Tools ready. Proceed to Step 4.


---
## Step 4 — Clone or update the repo

The render script lives in the repo. This step clones it if it is not already
present, or pulls the latest version if it is.

> Your `GITHUB_PAT` (Personal Access Token) is read from the encrypted master
> secrets file automatically — you do not need to paste it here.

In [4]:
import subprocess, os, sys

REPO_DIR  = '/content/ict-trading-bot'
REPO_URL  = 'https://github.com/the-lizardking/ict-trading-bot.git'

def shell(cmd, check=True, env=None):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, env=env)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.returncode != 0:
        if result.stderr.strip():
            print('Error:', result.stderr.strip())
        if check:
            sys.exit(result.returncode)
    return result

if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    print('Repo already cloned. Pulling latest ...')
    shell(f'git -C {REPO_DIR} pull --ff-only origin main')
else:
    print('Cloning repo ...')
    shell(f'git clone --depth 1 {REPO_URL} {REPO_DIR}')

print()
print(f'Repo ready at: {REPO_DIR}')
print('Proceed to Step 5.')

Cloning repo ...

Repo ready at: /content/ict-trading-bot
Proceed to Step 5.


---
## Step 5 — Choose your profile

Set `PROFILE` to the runtime you want to generate secrets for.

| Profile | Output file | Where it is used |
|---|---|---|
| `paper` | `.env.paper` | Local paper trading |
| `colab` | `.env.colab` | Google Colab sessions |
| `oracle_paper` | `.env.oracle_paper` | Oracle VM paper trading |
| `live` | `.env.live` | Oracle VM live trading |

> **Live trading warning:** Set `ALLOW_LIVE = True` only when you are
> intentionally generating credentials for live trading.
> The script will refuse to run the live profile without this flag.

In [5]:
# ── Choose your profile ───────────────────────────────────────────
PROFILE = 'paper'   # paper | colab | oracle_paper | live

# Set True ONLY when generating live trading credentials
ALLOW_LIVE = False
# ─────────────────────────────────────────────────────────────────

VALID_PROFILES = ('paper', 'colab', 'oracle_paper', 'live')

if PROFILE not in VALID_PROFILES:
    print(f'ERROR: Unknown profile "{PROFILE}". Choose one of: {VALID_PROFILES}')
    import sys; sys.exit(1)

if PROFILE == 'live' and not ALLOW_LIVE:
    print('ERROR: PROFILE is set to "live" but ALLOW_LIVE is False.')
    print('Set ALLOW_LIVE = True in this cell if you intend to generate live credentials.')
    import sys; sys.exit(1)

print(f'Profile selected : {PROFILE}')
if PROFILE == 'live':
    print('WARNING: You are about to generate LIVE trading credentials.')
print('Proceed to Step 6.')

Profile selected : paper
Proceed to Step 6.


---
## Step 6 — Generate the env file

This cell runs `render_env_from_master.py` from the repo.

- **No secret values are printed.**
- The output shows only the file path and the variable names written.
- The generated file is placed inside the repo directory in Colab's `/content`.
- You can then copy it to where your script needs it (e.g., the VM or a Colab env loader).

In [6]:
import subprocess, os, sys

SECRETS_DIR  = '/content/drive/MyDrive/ICT_Bot_Secrets'
MASTER_FILE  = os.path.join(SECRETS_DIR, 'master-secrets.sops.yaml')
AGE_KEY_FILE = os.path.join(SECRETS_DIR, 'age-keys.txt')
REPO_DIR     = '/content/ict-trading-bot'
SCRIPT       = os.path.join(REPO_DIR, 'scripts', 'render_env_from_master.py')

# Output path — inside the Colab repo workspace
profile_to_file = {
    'paper':        '.env.paper',
    'colab':        '.env.colab',
    'oracle_paper': '.env.oracle_paper',
    'live':         '.env.live',
}
OUT_FILE = os.path.join(REPO_DIR, profile_to_file[PROFILE])

cmd = [
    sys.executable,
    SCRIPT,
    '--master', MASTER_FILE,
    '--age-key-file', AGE_KEY_FILE,
    '--profile', PROFILE,
    '--out', OUT_FILE,
]

if PROFILE == 'live' and ALLOW_LIVE:
    cmd.append('--allow-live')

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
    env={**os.environ, 'PYTHONPATH': REPO_DIR},
)

# Print stdout (profile name, output path, variable names — no values)
if result.stdout.strip():
    print(result.stdout.strip())

if result.returncode != 0:
    # stderr may contain error messages; print as-is (no secret values in errors)
    if result.stderr.strip():
        print(result.stderr.strip())
    sys.exit(result.returncode)

print()
print(f'Env file generated: {OUT_FILE}')
print('Secrets were not printed. The file is ready to use.')

Profile : paper
Output  : /content/ict-trading-bot/.env.paper
Written : 21 variables
Keys    : ENVIRONMENT, EXCHANGE, MODE, DRY_RUN, ALLOW_LIVE_TRADING, TELEGRAM_BOT_TOKEN, TELEGRAM_CHAT_ID, BYBIT_TESTNET_API_KEY, BYBIT_TESTNET_API_SECRET, BYBIT_TESTNET_BASE_URL, HF_USERNAME, HF_TOKEN, SYMBOL, TIMEFRAME, DATA_DIR, MODEL_DIR, LOG_DIR, DB_PATH, MAX_POSITION_USD, MAX_DAILY_LOSS_USD, RISK_PER_TRADE

Env file generated: /content/ict-trading-bot/.env.paper
Secrets were not printed. The file is ready to use.


---
## What to do next

| Profile | Copy the file to |
|---|---|
| `paper` | The repo root on your local machine: `cp .env.paper /path/to/ict-trading-bot/.env.paper` |
| `colab` | A Colab cell that loads it: `from dotenv import load_dotenv; load_dotenv('.env.colab')` |
| `oracle_paper` | The Oracle VM via SCP: `scp .env.oracle_paper ubuntu@<VM_IP>:~/ict-trading-bot/.env.oracle_paper` |
| `live` | The Oracle VM via SCP: `scp .env.live ubuntu@<VM_IP>:~/ict-trading-bot/.env.live` |

> **Security:** Delete the env file from `/content` after copying it:
> ```python
> import os
> os.remove('/content/ict-trading-bot/.env.paper')  # adjust filename
> ```
> Disconnect the Colab runtime when done: **Runtime → Disconnect and delete runtime**.